In [1]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

d:\GEN-AI-Course\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\MZaid\AppData\Local\Temp\ipykernel_16520\2685459506.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [2]:
# Chat model
llm = ChatOllama(model="llama3.2:3b", temperature=0)

# Embedding model
embeddings = OllamaEmbeddings(model="nomic-embed-text")

## Step 1a - Indexing (Document Ingestion)


In [9]:
video_id = "aDG1T0kJnd4"
api = YouTubeTranscriptApi()
try:
    transcript_list = api.fetch(video_id)
    transcript=" ".join(item.text for item in transcript_list)
except TranscriptsDisabled:
    print("No captions available.")

In [11]:
print(transcript_list)

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='Thank you.', start=1.803, duration=1.0), FetchedTranscriptSnippet(text='Thank you.', start=2.803, duration=1.21), FetchedTranscriptSnippet(text='So, my father used to always tell me something\nwhich I want to share with you, that why do', start=4.013, duration=6.471), FetchedTranscriptSnippet(text='you want to fit inside a glass slipper?', start=10.484, duration=3.15), FetchedTranscriptSnippet(text='You know, like we were told, like Cinderella\ndid, why do you want to fit inside a glass', start=13.634, duration=4.76), FetchedTranscriptSnippet(text='slipper when you can shatter the glass ceiling?', start=18.394, duration=4.36), FetchedTranscriptSnippet(text='I want to tell you a little secret.', start=22.754, duration=2.479), FetchedTranscriptSnippet(text="I'm not very fond of this phrase, breaking\nthe glass ceiling.", start=25.233, duration=4.771), FetchedTranscriptSnippet(text='Why does it annoy me?', start=30.004, duration=1

## Step 1b - Indexing (Text Splitting)

In [18]:
text_splitters = RecursiveCharacterTextSplitter(
	chunk_size = 300,
	chunk_overlap = 0,
	separators=["\n\n", "\n", " ", ""],
)
result = text_splitters.create_documents([transcript])

In [19]:
for chunk in result:
  print(chunk)
  print("-------------------------------------------------")

page_content='Thank you. Thank you. So, my father used to always tell me something
which I want to share with you, that why do you want to fit inside a glass slipper? You know, like we were told, like Cinderella'
-------------------------------------------------
page_content='did, why do you want to fit inside a glass slipper when you can shatter the glass ceiling? I want to tell you a little secret. I'm not very fond of this phrase, breaking
the glass ceiling. Why does it annoy me? Because it takes the context of everything'
-------------------------------------------------
page_content='that I have done. All my achievements, all my hard work and
puts it into a box as if my ambition was that I want to find a glass ceiling and break it. Not at all. To be really honest, I was never on a mission'
-------------------------------------------------
page_content='to break, to shatter anything. All I wanted was to chase my dreams, my ambitions. I wanted to evolve. I wanted to become the best 

## Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [21]:
vector = FAISS.from_documents(result, embeddings)


In [22]:
vector.index_to_docstore_id

{0: '2be38f42-3758-405a-8e02-aa6428ab8d17',
 1: '097c8efb-2f55-4ae2-b3af-067e4b86786a',
 2: '7449ca97-ca6f-43bb-9c09-6ccebc1f2ef4',
 3: 'ee5a1b9c-8b1b-4c3a-9daa-a3e2dc473ad7',
 4: 'c3b6d660-4864-45cf-8de1-5852018d6581',
 5: 'f2c11f73-aa2c-4c3f-834c-1512059d5501',
 6: 'bb3bfcce-80e8-4626-a69a-947e77e29b89',
 7: '8658eec8-137b-4bbf-a587-6e6bfdcd50a1',
 8: '56b34463-edf8-449e-8866-4ae922c10fe0',
 9: '21e32422-cb8e-49d8-b98f-7743ecb457b3',
 10: '0c7200fc-69e2-4480-ac26-126d13ba4014',
 11: 'b5d47f4e-142f-482b-81da-c117002e5f88',
 12: '81c24a2a-b51d-4a20-9e4d-912226739a40',
 13: 'dd2d49a3-39b7-470a-a73d-8b550235f60e',
 14: '07b2edd5-8689-4ce7-9d99-fc0f0d4fc4b4',
 15: '5b3afad1-9e3a-471d-ac48-8d89b44ef61f',
 16: '6803dfee-ffa7-44a6-8af2-f8ee31dbf15e',
 17: '6f997d94-5f00-4def-8d3d-988e35b9458c',
 18: 'bfc2a009-c76d-45fd-adb6-234ca601e8f7',
 19: '5d7d3527-73bc-4b2b-9fe0-d9f8342ef768',
 20: '56e43998-2fc1-43ca-9c82-4a7403b8fdac',
 21: '03ac4914-4c0d-47e0-a242-191d4c0263b7',
 22: 'd9d6b3fb-5a83-

In [25]:
vector.get_by_ids(["51bbd47b-7366-4db8-a68b-9cce174ceefc"])

[Document(id='51bbd47b-7366-4db8-a68b-9cce174ceefc', metadata={}, page_content="remember where you came from. It's truly what defines you. So, I'm a proud Indian and army's daughter. Daughter of two doctors with a middle-class")]

## Step 2 - Retrieval

In [26]:
retrieval = vector.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [27]:
retrieval

VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000024D6A1E85F0>, search_kwargs={'k': 4})

In [28]:
retrieval.invoke("what is idea of this video")

[Document(id='097c8efb-2f55-4ae2-b3af-067e4b86786a', metadata={}, page_content="did, why do you want to fit inside a glass slipper when you can shatter the glass ceiling? I want to tell you a little secret. I'm not very fond of this phrase, breaking\nthe glass ceiling. Why does it annoy me? Because it takes the context of everything"),
 Document(id='bb3bfcce-80e8-4626-a69a-947e77e29b89', metadata={}, page_content='and members of minorities from rising beyond a certain level in a hierarchy. And this metaphor was first coined by feminists\nin reference to barriers in the careers of high achieving women. So, why did I choose it as my topic for today'),
 Document(id='6803dfee-ffa7-44a6-8af2-f8ee31dbf15e', metadata={}, page_content="fly. Give them wings, be who you want to be just\nby being fearless. Now, opportunities, that's another important\npart of being fearless. They are very funny thing, these opportunities,"),
 Document(id='ff0e3980-53e0-45bc-8295-2d4abf592bd8', metadata={}, page_c

## Step 3 - Augmentation